# Phase 2 Benchmark Runner (Colab)
Thin orchestration notebook. Training logic stays in `src/`.


In [ ]:
# Colab setup + repo bootstrap
import os
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -n "/content/drive/MyDrive/PlantVillage/Datasets.zip" -d "/content/"


if not os.path.isdir('/content/Plant-Disease-Detection-CV'):
    !git clone https://github.com/WilliamKyaww/Plant-Disease-Detection-CV.git
else:
    !cd /content/Plant-Disease-Detection-CV && git pull

!pip install -q -r /content/Plant-Disease-Detection-CV/requirements.txt

In [ ]:
# Canonical repo root
from pathlib import Path
import os, sys

repo_root = Path('/content/Plant-Disease-Detection-CV')  # cloned repo path
if not (repo_root / 'src').is_dir():
    raise FileNotFoundError(f"Repo not found at {repo_root}. Run clone cell first.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Using repo root:', repo_root)
print('Executed in Colab:', os.path.isdir('/content'))

In [ ]:
# Frozen split guard + dry-run benchmark
from pathlib import Path
manifest = Path('results/split_manifests/latest_split_manifest.json')
if manifest.exists():
    print('Using existing frozen splits (no regeneration).')
    print('Manifest:', manifest.resolve())
else:
    !python -m src.prepare_splits

!python -m src.run_phase2_benchmark --dry-run --no-pretrained --batch-size 4 --num-workers 0 --amp


In [ ]:
# Minimal training smoke (1 model, 1 seed, 1 epoch)
!python -m src.run_phase2_benchmark --models resnet18 --seeds 41 --epochs 1 --batch-size 32 --num-workers 2 --scheduler cosine --class-weighting inverse_frequency


In [ ]:
# Package key artifacts and download as one zip
from pathlib import Path
import zipfile
from google.colab import files

artifact_paths = [
    "results/phase2/model_seed_metrics.csv",
    "results/phase2/model_summary_mean_std.csv",
    "results/phase2/leaderboard.csv",
    "results/phase2/phase2_dry_run_report.json",
    "results/phase2/runs/resnet18/seed_41/metrics.json",
    "results/phase2/runs/resnet18/seed_41/phase2_resnet18_seed41.json",
    "results/colab_smoke/latest_colab_smoke.json",
    "results/colab_smoke/latest_colab_smoke.txt",
]

zip_name = "phase2_artifacts_export.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in artifact_paths:
        path = Path(p)
        if path.exists():
            zf.write(path, arcname=str(path))
        else:
            print(f"Missing (skipped): {p}")

files.download(zip_name)


In [ ]:
!python -m src.run_phase2_benchmark --models resnet18,resnet50,efficientnet_b0,vit_small_patch16_224 --seeds 41,42,43 --epochs 30 --batch-size 32 --num-workers 2 --scheduler cosine --class-weighting inverse_frequency --pretrained --amp --resume